# Pushover Analysis on MDOF Stick Model

## Introduction

This Jupyter Notebook outlines a structured workflow for conducting pushover analysis on multi-degree-of-freedom (MDOF) structural models using OpenSees. Pushover analysis is a static, nonlinear procedure that helps assess the inelastic behavior and capacity of structures under increasing lateral loads, simulating the effects of seismic demand in a simplified manner.

The analysis focuses on building models representative of typical structures located in seismic-prone regions, such as L'Aquila, and explores their performance through monotonic and cyclic pushover procedures. Unlike time-history analysis, which uses dynamic input, pushover analysis incrementally applies lateral forces or displacements to assess structural response and identify limit states.

The main goals of this notebook:

1. **Calibrate MDOF models based on single-degree-of-freedom (SDoF) oscillator capacity**: Calibrate storey-based force-deformation relationships using SDOF capacity curve definition (spectral displacement-spectral acceleration) based on the methodology of Lu et al. (2020) and other modifications to account for distinct response typologies (i.e., bilinear, trilinear and quadrilinear backbone definitions)

2. **Compile and construct MDOF Models in OpenSees**: Define and assemble MDOF models by specifying essential structural properties, including: mass, height, fundamental period, and the nonlinear behavior at each degree of freedom, including plastic hinges or other inelastic mechanisms

3. **Run Static (Monotonic) Pushover Analysis in OpenSees**: Apply a monotonically increasing lateral load (typically using a predefined force or displacement pattern such as inverted triangular or uniform) until structural collapse or a target displacement is reached. Extract key response metrics such as base shear vs. roof displacement/maximum interstorey drift curves (capacity curves)

4. **Run Cyclic Pushover Analysis in OpenSees**: Apply cyclic loading protocols to assess structural degradation, stiffness deterioration, and strength loss under repeated loading. This allows for evaluation of energy dissipation capacity, hysteretic response, etc.

## Initialize Libraries ##

In [ ]:
import numpy as np

from openquake.vmtk.units import units
from openquake.vmtk.modeller import modeller

## Calibrate and Compile the Stick Model ##

Below is a list of required input parameters associated with the single-degree-of-freedom oscillator if an SDOF-to-MDOF calibration is required.

In [ ]:
# Number of storeys
number_storeys = 3

# Relative floor heights list
floor_heights = [2.80, 3.00, 3.00]

# Relative floor masses list
floor_masses = [0.75, 0.75, 0.75]  # Unit mass for SDOFs

# MDOF capacity: 3 storeys × 4 points each (quadrilinear definition)
storey_drifts = np.array(
    [[0.008, 0.015, 0.025, 0.050],    # storey 1 drift capacity: Brittle failure at ~1.5% drift
     [0.010, 0.018, 0.028, 0.055],    # storey 2 drift capacity
     [0.012, 0.022, 0.032, 0.060]])   # storey 3 drift capacity


storey_forces = np.array(
    [[3.5, 5.0, 1.5, 1.5],            # storey 1 shear force: Sharp drop from 10kN to 3kN
     [4.0, 5.5, 1.7, 1.7],            # storey 2 shear force
     [4.5, 6.5, 2.0, 2.0]]) * units.kN  # storey 3 shear force


# Flag to activate default stiffness-strength degradation and pinching4
mdof_degradation = True

## Setting Up, Running and Exporting Static (Monotonic) Pushover Analysis ##

The `do_spo_analysis` method performs a displacement-controlled static (monotonic) pushover analysis on the compiled MDOF stick model. A lateral load pattern is incrementally applied floor-by-floor up to a user-defined target displacement, and the structural response — base shear, floor displacements, storey drift ratios, and element forces — is recorded at every step.

The following parameters are required to run a static pushover analysis using the `do_spo_analysis` method:

- **`ref_disp`**: The reference (seed) displacement in metres from which the analysis is incremented — typically the yield displacement or a small fraction thereof (e.g. `0.005` m).
- **`disp_scale_factor`**: A multiplier applied to `ref_disp` to define the total target displacement. For example, `ref_disp = 0.005` and `disp_scale_factor = 15` pushes the structure to `0.075` m.
- **`push_dir`**: Integer flag controlling the direction of the lateral load. Set to `1` for X-direction, `2` for Y-direction, or `3` for Z-direction.
- **`phi`**: The storey load-pattern shape vector, scaled by floor mass internally. This is typically the first-mode shape from `do_modal_analysis` or a hand-defined triangular pattern such as `[0.33, 0.66, 1.00]` for a three-storey model.
- **`pFlag`**: Boolean flag to toggle step-by-step solver output to the console. Set to `True` for verbose feedback or `False` to suppress printing. Defaults to `True`.
- **`num_steps`**: The number of equal displacement increments used to reach the target displacement. A finer step count improves curve resolution but increases runtime. Defaults to `200`.
- **`ansys_soe`**: The OpenSees system-of-equations solver. Defaults to `'BandGeneral'`, which is suitable for most banded systems arising from stick models.
- **`constraints_handler`**: The method used to enforce constraint equations in the model. Defaults to `'Transformation'`.
- **`numberer`**: The degree-of-freedom numbering algorithm. Defaults to `'RCM'` (Reverse Cuthill–McKee), which minimises matrix bandwidth.
- **`test_type`**: The convergence test applied at each step. Defaults to `'EnergyIncr'`, which checks the inner product of the displacement increment and unbalanced force.
- **`init_tol`**: The convergence tolerance threshold. Defaults to `1e-5`.
- **`init_iter`**: The maximum number of Newton iterations allowed per step before the solver declares non-convergence. Defaults to `1000`.
- **`algorithm_type`**: The nonlinear solution algorithm. Defaults to `'KrylovNewton'`, which uses a Krylov subspace accelerator for improved convergence under strength degradation.
- **`save_animation_path`**: Optional file path for the exported pushover animation. Supported formats are `.gif` and `.mp4`. Set to `None` to skip animation entirely.

The method returns a dictionary `results` with the following keys:

- **`spo_disps`**: Array of shape `(steps, floors)` containing the lateral displacement at every floor node at each analysis step [m].
- **`spo_rxn`**: Array of shape `(steps,)` containing the base shear reaction recorded at the fixed base [kN].
- **`spo_disps_spring`**: Array of shape `(steps, storeys)` containing the deformation of each zero-length spring element [m].
- **`spo_forces_spring`**: Array of shape `(steps, storeys)` containing the shear force in each zero-length spring element [kN].
- **`spo_idr`**: Array of shape `(steps, storeys)` containing the inter-storey drift ratio history for every storey.
- **`spo_midr`**: Array of shape `(steps,)` containing the maximum IDR across all storeys at every step.

When `save_animation_path` is provided, an animation is automatically generated and exported alongside the results. The animation shows three synchronised panels updating frame-by-frame: the **deformed model shape** on the left, the evolving **pushover curve** (base shear vs. roof displacement) on the top right, and the **base shear vs. maximum inter-storey drift ratio** curve on the bottom right. A preview of the animation produced by the cell below is displayed at the end of this section.

In [ ]:
# Compile the MDOF model
model = modeller(number_storeys,
                 floor_heights,
                 floor_masses,
                 storey_drifts,
                 storey_forces,
                 # No need to multiply by g since the units are already in kN
                 mdof_degradation)         # Initialise the class (Build the model)
model.compile_model()                      # Compile the MDOF model

# View the model
model.plot_model()                         # Visualise the model

# Do gravity analysis
model.do_gravity_analysis()                # Do gravity analysis

# Do modal analysis
T, phi = model.do_modal_analysis(num_modes=number_storeys,
                                 plot_modes=False)   # Do modal analysis and get period of vibration

# Define pushover analysis parameters
ref_disp = 0.005                 # Reference displacement
disp_scale_factor = 15           # Multiplier of the reference displacement
push_dir = 1                     # Push direction (for X-direction:1, for Y-direction=2)
# Load pattern (In this example, we apply a user-defined triangular pattern)
phi = [0.33, 0.66, 1.00]

# Do pushover analysis
results = model.do_spo_analysis(
    ref_disp,
    disp_scale_factor,
    push_dir,
    phi,
    pFlag=False,
    num_steps=200,
    ansys_soe='BandGeneral',
    constraints_handler='Transformation',
    numberer='RCM',
    test_type='EnergyIncr',
    init_tol=1.0e-5,
    init_iter=1000,
    algorithm_type='KrylovNewton',
    # Export animation of SPO when setting path, set to None if you opt not to
    save_animation_path='out/spo_animation.gif')

print('ANALYSIS COMPLETED!')

## Explore the Results of the Static Pushover Analysis ##

In [4]:
# Unpack the SPO results
spo_disps = results['spo_disps']          # Displacements at each control node (i.e., floor)
spo_rxn = results['spo_rxn']            # Reaction at base (Base shear)
# Displacements in nonlinear springs (i.e., zero-length elements)
spo_disps_spring = results['spo_disps_spring']
# Forces in nonlinear springs (i.e., zero-length elements)
spo_forces_spring = results['spo_forces_spring']
spo_idr = results['spo_idr']            # Interstorey drift ratios
spo_midr = results['spo_midr']           # Maximum interstorey drifts

![spo_animation](out/spo_animation.gif)

## Setting Up, Running and Exporting Cyclic Pushover Analysis ##

The `do_cpo_analysis` method performs a displacement-controlled cyclic pushover analysis on the compiled MDOF stick model. The structure is pushed and pulled through a sequence of increasing ductility targets, and the full hysteretic response — base shear, floor displacements, storey drift ratios, and cumulative dissipated energy — is recorded at every step.

The following parameters are required to run a cyclic pushover analysis using the `do_cpo_analysis` method:

- **`ref_disp`**: The reference (yield) displacement in metres used to scale each ductility cycle (e.g. `0.001` m). Each cycle target is computed as `mu × ref_disp`, where `mu` is drawn from `mu_levels`.
- **`mu_levels`**: A list of ductility multipliers defining the peak displacement of each cycle pair. For example, `[1, 2, 4, 6, 8, 10]` produces cycles to ±1×, ±2×, ±4×, ±6×, ±8×, and ±10× the reference displacement, in that order.
- **`push_dir`**: Integer flag controlling the direction of the lateral load. Set to `1` for X-direction, `2` for Y-direction.
- **`dispIncr`**: The number of equal displacement increments used within each half-cycle. A larger value improves hysteresis loop resolution but increases runtime.
- **`phi`**: The storey load-pattern shape vector, scaled by floor mass internally. This is typically based on the first-mode shape from `do_modal_analysis` or a hand-defined triangular pattern such as `[0.33, 0.66, 1.00]` for a three-storey model.
- **`pFlag`**: Boolean flag to toggle step-by-step solver output to the console. Set to `True` for verbose feedback or `False` to suppress printing. Defaults to `True`.
- **`ansys_soe`**: The OpenSees system-of-equations solver. Defaults to `'BandGeneral'`, which is suitable for most banded systems arising from stick models.
- **`constraints_handler`**: The method used to enforce constraint equations in the model. Defaults to `'Transformation'`.
- **`numberer`**: The degree-of-freedom numbering algorithm. Defaults to `'RCM'` (Reverse Cuthill–McKee), which minimises matrix bandwidth.
- **`test_type`**: The convergence test applied at each step. Defaults to `'NormDispIncr'`, which checks the norm of the displacement increment vector.
- **`init_tol`**: The convergence tolerance threshold. Defaults to `1e-5`.
- **`init_iter`**: The maximum number of Newton iterations allowed per step before the solver declares non-convergence. Defaults to `1000`.
- **`algorithm_type`**: The nonlinear solution algorithm. Defaults to `'KrylovNewton'`, which uses a Krylov subspace accelerator for improved convergence under strength and stiffness degradation.
- **`save_animation_path`**: Optional file path for the exported cyclic animation. Supported formats are `.gif` and `.mp4`. Set to `None` to skip animation entirely.

The method returns a dictionary `results` with the following keys:

- **`cpo_disps`**: Array of shape `(steps, floors)` containing the lateral displacement at every floor node at each analysis step [m].
- **`cpo_rxn`**: Array of shape `(steps,)` containing the base shear reaction recorded at the fixed base [kN].
- **`cpo_idr`**: Array of shape `(steps, storeys)` containing the inter-storey drift ratio history for every storey.
- **`cpo_energy`**: Array of shape `(steps, 2)` where column 0 is the pseudo-step index and column 1 is the cumulative dissipated energy [kN·m].

When `save_animation_path` is provided, an animation is automatically generated and exported alongside the results. The animation shows four synchronised panels updating frame-by-frame: the **deformed model shape** on the left, the evolving **hysteretic curve** (base shear vs. roof displacement) on the top right, the **base shear vs. maximum inter-storey drift ratio** hysteresis loop on the center right, and the **cumulative dissipated energy vs analysis steps** on the bottom right. A preview of the animation produced by the cell below is displayed at the end of this section.

In [ ]:
# Compile the MDOF model
model = modeller(number_storeys,
                 floor_heights,
                 floor_masses,
                 storey_drifts,
                 storey_forces,
                 mdof_degradation)         # Initialise the class (Build the model)
model.compile_model()                      # Compile the MDOF model

# View the model
model.plot_model()                         # Visualise the model

# Do gravity analysis
model.do_gravity_analysis()                # Do gravity analysis

# Do modal analysis
T, phi = model.do_modal_analysis(num_modes=number_storeys,
                                 plot_modes=False)   # Do modal analysis and get period of vibration

# Define pushover analysis parameters
ref_disp = 0.012                                       # Reference displacement
mu_levels = [0.5, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 10]   # Target ductility factor
# The number of displacement increments for each loading cycle
dispIncr = 10
max_step = ref_disp * 0.05                 # No step larger than 5% of yield displacement
# Load pattern (In this example, we apply a user-defined triangular pattern)
phi = [0.33, 0.66, 1.00]

# Do pushover analysis
results = model.do_cpo_analysis(
    ref_disp,
    mu_levels,
    push_dir,
    dispIncr,
    phi,
    max_step=max_step,
    pFlag=False,
    ansys_soe='BandGeneral',
    constraints_handler='Transformation',
    numberer='RCM',
    test_type='NormDispIncr',
    init_tol=1.0e-5,
    init_iter=1000,
    algorithm_type='KrylovNewton',
    # Export animation of CPO when setting path, set to None if you opt not to
    save_animation_path='out/cpo_animation.gif')

print('ANALYSIS COMPLETED!')

## Explore the Results of the Cyclic Pushover Analysis ## 

In [6]:
# Unpack results
cpo_disps = results['cpo_disps']  # Displacements at each control node (i.e., floor)
cpo_rxn = results['cpo_rxn']    # Reaction at base (Base shear)
cpo_energy = results['cpo_energy']  # Cumulative Dissipated Energy

![cpo_animation](out/cpo_animation.gif)